# Module 4: Advanced RAG & Vector Engineering
## Task 4: The "Memory Vault" (Advanced RAG with Citations)

### **Goal**
Build a RAG system that doesn't hallucinate.
1. **Hybrid Retrieval:** Using vector search for meaning.
2. **Reranking:** Implementing a Cross-Encoder to select the highest-quality chunks.
3. **Citations:** Forcing the agent to cite its sources by page and filename.

In [5]:
# !pip install langchain-mistralai langchain-community faiss-cpu flashrank pypdf

import os
from langchain_mistralai import ChatMistralAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate

# --- THE FIX: Updated Import Paths for 2026 ---
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import FlashrankRerank
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings

# Configuration
os.environ["MISTRAL_API_KEY"] = os.getenv("MISTRAL_API_KEY", "your-key-here")
llm = ChatMistralAI(model="mistral-large-latest", temperature=0)

# --- THE FIX: Initialize the Embeddings Model ---
embeddings = MistralAIEmbeddings(model="mistral-embed")

### **The Memory Vault: Chunking and Vectorization**
We load our PDF documents and split them into smaller, overlapping "chunks". This ensures that the context is preserved even if a sentence is split between two pieces of data.

In [6]:
# Load your medical or technical PDF
loader = PyPDFLoader("Mindgigs AI(Python) Course.pdf")
docs = loader.load()

# For demonstration, we use a mock text splitter logic
text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
chunks = text_splitter.split_documents(docs)

# Initialize Embeddings and FAISS Vector Store
# Note: In a real task, replace with actual document chunks
vectorstore = FAISS.from_documents(chunks, embeddings)

### **Context Distillation (Reranking)**
Vector search is fast but "fuzzy". We use **FlashrankRerank** to take the top 10 results and mathematically rank them against the query to find the top 3 most relevant segments.

In [7]:
# Initialize the Reranker
compressor = FlashrankRerank()

# Build the Compressed Retriever
# base_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
# compression_retriever = ContextualCompressionRetriever(
#    base_compressor=compressor, base_retriever=base_retriever
# )

INFO:flashrank.Ranker:Downloading ms-marco-MultiBERT-L-12...
ms-marco-MultiBERT-L-12.zip: 100%|██████████| 98.7M/98.7M [00:27<00:00, 3.76MiB/s]


### **Implementing Citations**
We wrap the retrieval logic into a **Tool**. This allows our Agent to choose to "Search the Memory Vault" whenever it needs factual information.

In [10]:
@tool
def query_memory_vault(question: str):
    """Searches the internal PDF database for topics related to the question."""
    # Simulation of retrieval and reranking
    # In production, you would call: compression_retriever.get_relevant_documents(question)
    
    print(f"\n--- [RAG LOG]: Reranking 10 chunks for query: {question} ---")
    context = (
        "1. [Source:pdf, How can and where i can learn this course .\n"
        "2. [Source: pdf about the python course]: The course covers advanced Python programming concepts."
    )
    return context

tools = [query_memory_vault]

# Define the Agent using the unified 2026 factory
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a precise technical assistant. Always cite your source and page number."
)

In [11]:
# Launch the Agentic RAG request
query = "What does the course say  and what are the deployment requirements?"

input_payload = {
    "messages": [
        {"role": "user", "content": query}
    ]
}

response = agent.invoke(input_payload)

print("\n--- FINAL ANSWER WITH CITATIONS ---")
print(response["messages"][-1].content)

INFO:httpx:HTTP Request: POST https://api.mistral.ai/v1/chat/completions "HTTP/1.1 200 OK"



--- [RAG LOG]: Reranking 10 chunks for query: What are the course details and objectives in the document? ---

--- [RAG LOG]: Reranking 10 chunks for query: What are the deployment requirements in the document? ---


INFO:httpx:HTTP Request: POST https://api.mistral.ai/v1/chat/completions "HTTP/1.1 200 OK"



--- FINAL ANSWER WITH CITATIONS ---
Here is the detailed information based on the available sources:

---

### **Course Details**
The course focuses on **advanced Python programming concepts**. However, the specific topics covered, duration, prerequisites, and learning objectives are not explicitly detailed in the retrieved information.
**Source:** Internal PDF (Python course material).

If you need precise details about the syllabus or course structure, I recommend checking the following:
- The course platform (e.g., Coursera, Udemy, edX, or an institutional LMS).
- The official course documentation or brochure.

---

### **Deployment Requirements**
The retrieved documents **do not explicitly mention deployment requirements** for the course or its projects. However, based on general best practices for deploying Python applications, here are the typical requirements:

---

#### **1. Software Requirements**
- **Python Version**: Python 3.8 or higher (recommended).
  *Source: Industry s